# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassanKh4lil/ML-WEEK-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Lane: **CTR-vs-position review** (carried over from `w03_data_contract.ipynb`). Uses the starter CSV — no HF token needed.

> Working with an AI assistant? Read `skills/README.md`, then load `building-baselines` + `flyrank/flyrank-data`.

## 1. Two signal checks + my rule in plain words

**The rule, in one paragraph:** A page is worth a CTR-review if it's getting real search visibility
(enough impressions to trust the number) but its click-through rate sits well below what pages *at
its own position* typically get. That's a snippet/title problem, not a ranking problem — and it's
worth fixing first on the pages with the most impressions at stake, because that's where a CTR
fix returns the most clicks.

**Signal 1 — CTR vs. position (flag-linked: this is the exact signal behind the session's CTR-fix
logic).** If CTR doesn't actually vary by position tier, "below the tier median" is meaningless —
this has to be checked before it can anchor a rule.

**Signal 2 — Impression volume vs. clicks (flag-linked: the volume signal behind quick-win logic).**
The rule multiplies the CTR gap by impression volume to prioritize. That only makes sense if higher
impression tiers really do carry proportionally more click volume at stake — checked below.

Data-quality note applied to both: `avg_position == 0` means *no data*, not rank zero (1,205 rows
in this slice) — and it turns out those rows are mis-tagged `top_3` in `position_tier`, so they're
dropped before any bucket table. Per the data dictionary's own warning, tier medians are only
trustworthy above a volume floor (`impressions_90d >= 100`), so that floor is applied too.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print("raw rows:", len(df))

# avg_position == 0 means "no data" (data dictionary gotcha), and these rows
# turn out to be mis-tagged 'top_3' in position_tier -- drop before any bucket table.
n_zero_pos = (df['avg_position'] == 0).sum()
print("rows with avg_position == 0 (dropped, mis-tagged top_3):", n_zero_pos)
df = df[df['avg_position'] > 0].copy()
print("rows after dropping no-data positions:", len(df))

raw rows: 30000
rows with avg_position == 0 (dropped, mis-tagged top_3): 1205


rows after dropping no-data positions: 28795


In [2]:
# --- Signal 1: CTR vs. position tier ---------------------------------------
VOL_FLOOR = 100  # data dictionary: tier medians need a volume floor to be trustworthy
tier_order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']

sig1 = (
    df[df['impressions_90d'] >= VOL_FLOOR]
    .groupby('position_tier', observed=True)['ctr']
    .agg(n='count', median_ctr='median', mean_ctr='mean')
    .reindex(tier_order)
)
print(f"Signal 1 — median CTR by position tier (impressions_90d >= {VOL_FLOOR}, n printed per bucket)")
print(sig1)

verdict_1 = "CONFIRMED"
print(f"\nVerdict: {verdict_1} — median CTR is highest for top_3/page_1 (0.19-0.23) and steps down "
      "through striking (0.15) and page_3_5 (0.06) to deep (0.00). The tier ordering isn't perfectly "
      "monotonic at the very top (top_3 slightly below page_1, n=533 vs 8,633 -- a small, noisy bucket), "
      "but the overall decline with worse position is clear and large enough to anchor a rule on.")

Signal 1 — median CTR by position tier (impressions_90d >= 100, n printed per bucket)
                  n  median_ctr  mean_ctr
position_tier                            
top_3           533        0.19  0.334128
page_1         8633        0.23  0.354760
striking       5903        0.15  0.255782
page_3_5       6058        0.06  0.142359
deep            879        0.00  0.055415

Verdict: CONFIRMED — median CTR is highest for top_3/page_1 (0.19-0.23) and steps down through striking (0.15) and page_3_5 (0.06) to deep (0.00). The tier ordering isn't perfectly monotonic at the very top (top_3 slightly below page_1, n=533 vs 8,633 -- a small, noisy bucket), but the overall decline with worse position is clear and large enough to anchor a rule on.


In [3]:
# --- Signal 2: impression volume vs. clicks at stake -----------------------
impression_order = ['low', 'moderate', 'good', 'excellent']

sig2 = (
    df.groupby('impression_tier', observed=True)
      .agg(n=('content_id', 'count'),
           median_ctr=('ctr', 'median'),
           mean_clicks_last_30d=('clicks_last_30d', 'mean'),
           mean_impressions_last_30d=('impressions_last_30d', 'mean'))
      .reindex(impression_order)
)
print("Signal 2 — clicks at stake by impression tier (n printed per bucket)")
print(sig2)

verdict_2 = "CONFIRMED"
print(f"\nVerdict: {verdict_2} — mean clicks-last-30d rise sharply and monotonically from 'low' (0.06) "
      "to 'moderate' (0.82) to 'good' (9.2) to 'excellent' (67.1) as impression tier rises. Higher-"
      "impression content really does carry proportionally more click volume at stake, so weighting the "
      "CTR gap by impressions (instead of treating every underperforming page equally) is justified: a "
      "fixed-size CTR fix returns far more clicks on a high-impression page than a low-impression one.")

Signal 2 — clicks at stake by impression tier (n printed per bucket)
                     n  median_ctr  mean_clicks_last_30d  \
impression_tier                                            
low              10043        0.00              0.062133   
moderate         10469        0.12              0.824912   
good              7205        0.21              9.216794   
excellent         1078        0.22             67.112245   

                 mean_impressions_last_30d  
impression_tier                             
low                              17.489595  
moderate                        276.428312  
good                           2461.152949  
excellent                     20470.737477  

Verdict: CONFIRMED — mean clicks-last-30d rise sharply and monotonically from 'low' (0.06) to 'moderate' (0.82) to 'good' (9.2) to 'excellent' (67.1) as impression tier rises. Higher-impression content really does carry proportionally more click volume at stake, so weighting the CTR gap by impressi

## 2. Build the ranked queue (writes the CSV)

**Score:** `score = max(0, tier_median_ctr - ctr) * impressions_last_30d`, computed only for rows
with `impressions_90d >= 100` (the same visibility floor used above — below it the tier median
isn't trustworthy and neither is the row's own CTR).

**Reason code (one):** `ctr_below_tier_median`

**Action label:** `review_meta_and_snippet` when the gap is positive and the row is eligible;
`no_action` otherwise.

No future-window or label-derived inputs: only `position_tier`, `ctr`, and last-30d impressions —
no `trend_direction`/`trend_pct` (label source, per the data dictionary), no `prev_30d` columns.

In [4]:
VOL_FLOOR = 100

tier_median_ctr = (
    df[df['impressions_90d'] >= VOL_FLOOR]
    .groupby('position_tier', observed=True)['ctr']
    .median()
)

df['tier_median_ctr'] = df['position_tier'].map(tier_median_ctr)
eligible = df['impressions_90d'] >= VOL_FLOOR

df['ctr_gap'] = (df['tier_median_ctr'] - df['ctr']).clip(lower=0)
df['score'] = np.where(eligible, df['ctr_gap'] * df['impressions_last_30d'], 0.0)
df['reason_code'] = 'ctr_below_tier_median'
df['action'] = np.where(df['score'] > 0, 'review_meta_and_snippet', 'no_action')

ranked = df.sort_values('score', ascending=False).reset_index(drop=True)

queue_cols = ['content_id', 'client_id', 'position_tier', 'ctr', 'tier_median_ctr',
              'ctr_gap', 'impressions_last_30d', 'score', 'reason_code', 'action']
queue = ranked[queue_cols].copy()

import os
os.makedirs('../outputs', exist_ok=True)
queue.to_csv('../outputs/baseline_action_score.csv', index=False)

print("queue rows written:", len(queue))
print("rows flagged for action:", (queue['action'] == 'review_meta_and_snippet').sum())
print("base rate (share flagged):", round((queue['action'] == 'review_meta_and_snippet').mean(), 3))
queue.head(10)

queue rows written: 28795
rows flagged for action: 10126
base rate (share flagged): 0.352


,content_id,client_id,position_tier,ctr,tier_median_ctr,ctr_gap,impressions_last_30d,score,reason_code,action
0,content_8451fc6f034d,client_d029fa3a95,top_3,0.03,0.19,0.16,168958,27033.28,ctr_below_tier_median,review_meta_and_snippet
1,content_4a6607efcb46,client_6208ef0f77,top_3,0.01,0.19,0.18,122303,22014.54,ctr_below_tier_median,review_meta_and_snippet
2,content_c84a0ab98e90,client_f369cb89fc,page_1,0.03,0.23,0.20,99337,19867.40,ctr_below_tier_median,review_meta_and_snippet
3,content_36ff89c8214e,client_19581e27de,page_1,0.05,0.23,0.18,106985,19257.30,ctr_below_tier_median,review_meta_and_snippet
4,content_c8e9d6ab9013,client_19581e27de,page_1,0.00,0.23,0.23,63326,14564.98,ctr_below_tier_median,review_meta_and_snippet
5,content_91652435f57a,client_19581e27de,page_1,0.06,0.23,0.17,74135,12602.95,ctr_below_tier_median,review_meta_and_snippet
6,content_73c54f78c06a,client_f369cb89fc,page_1,0.10,0.23,0.13,85885,11165.05,ctr_below_tier_median,review_meta_and_snippet
7,content_5fe46e04994d,client_4e07408562,page_1,0.14,0.23,0.09,120791,10871.19,ctr_below_tier_median,review_meta_and_snippet
8,content_b115f7c74779,client_19581e27de,page_1,0.03,0.23,0.20,51115,10223.00,ctr_below_tier_median,review_meta_and_snippet
9,content_4c76e9b13aea,client_19581e27de,page_1,0.07,0.23,0.16,55948,8951.68,ctr_below_tier_median,review_meta_and_snippet


## 3. Top-10 review

For each of the top 10 by score: the action, why it's there, and what would make it wrong.

In [5]:
top10 = queue.head(10).reset_index(drop=True)
for i, row in top10.iterrows():
    print(f"{i+1}. {row['content_id']} | action={row['action']} | "
          f"tier={row['position_tier']} | ctr={row['ctr']:.2f} vs tier median {row['tier_median_ctr']:.2f} | "
          f"impressions_last_30d={row['impressions_last_30d']:.0f} | score={row['score']:.0f}")
top10

1. content_8451fc6f034d | action=review_meta_and_snippet | tier=top_3 | ctr=0.03 vs tier median 0.19 | impressions_last_30d=168958 | score=27033
2. content_4a6607efcb46 | action=review_meta_and_snippet | tier=top_3 | ctr=0.01 vs tier median 0.19 | impressions_last_30d=122303 | score=22015
3. content_c84a0ab98e90 | action=review_meta_and_snippet | tier=page_1 | ctr=0.03 vs tier median 0.23 | impressions_last_30d=99337 | score=19867
4. content_36ff89c8214e | action=review_meta_and_snippet | tier=page_1 | ctr=0.05 vs tier median 0.23 | impressions_last_30d=106985 | score=19257
5. content_c8e9d6ab9013 | action=review_meta_and_snippet | tier=page_1 | ctr=0.00 vs tier median 0.23 | impressions_last_30d=63326 | score=14565
6. content_91652435f57a | action=review_meta_and_snippet | tier=page_1 | ctr=0.06 vs tier median 0.23 | impressions_last_30d=74135 | score=12603
7. content_73c54f78c06a | action=review_meta_and_snippet | tier=page_1 | ctr=0.10 vs tier median 0.23 | impressions_last_30d=8588

,content_id,client_id,position_tier,ctr,tier_median_ctr,ctr_gap,impressions_last_30d,score,reason_code,action
0,content_8451fc6f034d,client_d029fa3a95,top_3,0.03,0.19,0.16,168958,27033.28,ctr_below_tier_median,review_meta_and_snippet
1,content_4a6607efcb46,client_6208ef0f77,top_3,0.01,0.19,0.18,122303,22014.54,ctr_below_tier_median,review_meta_and_snippet
2,content_c84a0ab98e90,client_f369cb89fc,page_1,0.03,0.23,0.20,99337,19867.40,ctr_below_tier_median,review_meta_and_snippet
3,content_36ff89c8214e,client_19581e27de,page_1,0.05,0.23,0.18,106985,19257.30,ctr_below_tier_median,review_meta_and_snippet
4,content_c8e9d6ab9013,client_19581e27de,page_1,0.00,0.23,0.23,63326,14564.98,ctr_below_tier_median,review_meta_and_snippet
5,content_91652435f57a,client_19581e27de,page_1,0.06,0.23,0.17,74135,12602.95,ctr_below_tier_median,review_meta_and_snippet
6,content_73c54f78c06a,client_f369cb89fc,page_1,0.10,0.23,0.13,85885,11165.05,ctr_below_tier_median,review_meta_and_snippet
7,content_5fe46e04994d,client_4e07408562,page_1,0.14,0.23,0.09,120791,10871.19,ctr_below_tier_median,review_meta_and_snippet
8,content_b115f7c74779,client_19581e27de,page_1,0.03,0.23,0.20,51115,10223.00,ctr_below_tier_median,review_meta_and_snippet
9,content_4c76e9b13aea,client_19581e27de,page_1,0.07,0.23,0.16,55948,8951.68,ctr_below_tier_median,review_meta_and_snippet


**Line-by-line (action / why / what would make it wrong):**

1. **review_meta_and_snippet** — top-ranked `top_3` page, CTR far below the top_3 median despite
   heavy impressions. *Wrong if:* the low CTR is because the query is navigational/branded and
   users are clicking a different (correct) result on purpose, not skipping a bad snippet.
2. **review_meta_and_snippet** — same pattern, `top_3`, huge impression volume driving the score.
   *Wrong if:* this URL has a redirect or canonical issue depressing clicks that a snippet rewrite
   won't fix.
3. **review_meta_and_snippet** — `page_1`, big CTR gap and impressions. *Wrong if:* the page already
   had a snippet test running and this is a temporary dip, not a stable pattern.
4. **review_meta_and_snippet** — `page_1`, similar profile. *Wrong if:* competition_level on this
   query is unusually high (many strong competing snippets) so the ceiling CTR here is genuinely
   lower than the tier median, not a fixable problem.
5. **review_meta_and_snippet** — `page_1`, large gap. *Wrong if:* `main_intent` is informational and
   users are satisfied by the SERP snippet/answer box alone (no-click search), which a rewrite can't
   change.
6. **review_meta_and_snippet** — `page_1`, high impressions. *Wrong if:* seasonal/trend spike inflated
   `impressions_last_30d` this period only, so the score overstates ongoing opportunity.
7. **review_meta_and_snippet** — `page_1`. *Wrong if:* `content_type` is something like a listing/
   category page where title rewrites are constrained by a template, limiting real fix options.
8. **review_meta_and_snippet** — `page_1`. *Wrong if:* the page is very new (`content_age_days` low)
   and CTR simply hasn't stabilized yet.
9. **review_meta_and_snippet** — `page_1`. *Wrong if:* word_count/content mismatch means the title
   already accurately reflects thin content — a snippet fix would just inflate false-promise clicks.
10. **review_meta_and_snippet** — `page_1`. *Wrong if:* this content shares a template with several
    other pages and a rewrite would need to be applied site-wide, changing the cost/benefit of the
    fix.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# Weak-pick candidates: large score driven almost entirely by volume, with a modest CTR gap --
# i.e. the ranking may be more "big page" than "clear CTR problem."
queue['gap_ratio'] = queue['ctr_gap'] / queue['tier_median_ctr'].replace(0, np.nan)
weak = queue[queue['action'] == 'review_meta_and_snippet'].sort_values('gap_ratio').head(5)
print("Weak-pick candidates (small relative CTR gap despite making the flagged list):")
print(weak[['content_id', 'position_tier', 'ctr', 'tier_median_ctr', 'gap_ratio', 'score']])

print("\nThese are weak because a small relative gap on a tier with a low median CTR to begin with "
      "can still produce a nonzero score once multiplied by impressions -- volume is doing most of "
      "the work here, not a real CTR problem. A confidence threshold on gap_ratio (e.g. >= 0.3) would "
      "filter these out in a v2 of the rule.")

# Leakage check: confirm none of the label-source / future-window columns were used as inputs.
banned_cols = ['trend_direction', 'trend_pct', 'is_declining_label']
used_cols = ['position_tier', 'ctr', 'impressions_last_30d', 'impressions_90d']
print("\nBanned (label-derived) columns used as inputs:",
      [c for c in banned_cols if c in used_cols] or "none")
print("Future-window columns used as inputs:", [c for c in used_cols if 'prev' in c] or "none")

Weak-pick candidates (small relative CTR gap despite making the flagged list):
                content_id position_tier   ctr  tier_median_ctr  gap_ratio  \
8197  content_33893ed00144        page_1  0.22             0.23   0.043478   
8113  content_f6a58fc8b0d1        page_1  0.22             0.23   0.043478   
5772  content_3822b7c89316        page_1  0.22             0.23   0.043478   
5747  content_79aa5aa9e4a7        page_1  0.22             0.23   0.043478   
5044  content_034f3b74fa3c        page_1  0.22             0.23   0.043478   

      score  
8197   4.32  
8113   4.51  
5772  12.75  
5747  12.94  
5044  16.99  

These are weak because a small relative gap on a tier with a low median CTR to begin with can still produce a nonzero score once multiplied by impressions -- volume is doing most of the work here, not a real CTR problem. A confidence threshold on gap_ratio (e.g. >= 0.3) would filter these out in a v2 of the rule.

Banned (label-derived) columns used as inputs: none

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.